In [6]:
import cv2
import os

def extract_frames(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    for class_name in os.listdir(input_dir):
        class_path = os.path.join(input_dir, class_name)
        output_class_path = os.path.join(output_dir, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for video_file in os.listdir(class_path):
            video_path = os.path.join(class_path, video_file)
            cap = cv2.VideoCapture(video_path)

            count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame = cv2.resize(frame, (150,150))
                frame_name = f"{video_file}_{count}.jpg"
                cv2.imwrite(os.path.join(output_class_path, frame_name), frame)
                count += 1

            cap.release()

# Run for both train & test
extract_frames("data/train", "data_frames/train")
extract_frames("data/test", "data_frames/test")

In [5]:
import os
import shutil

source_dir = "data/test"

for file in os.listdir(source_dir):
    if file.endswith(".avi"):
        class_name = file.split("_")[1]   # CricketShot
        
        class_path = os.path.join(source_dir, class_name)
        os.makedirs(class_path, exist_ok=True)
        
        shutil.move(
            os.path.join(source_dir, file),
            os.path.join(class_path, file)
        )

print("Organized successfully ✅")

Organized successfully ✅


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    "data_frames/train",
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical'
)

test_data = test_gen.flow_from_directory(
    "data_frames/test",
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical'
)

Found 119140 images belonging to 6 classes.
Found 39745 images belonging to 6 classes.


In [8]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

c:\Users\SwapnilShinde\OneDrive\Documents\Study Material\Sem 6\AML\Experiment Codes\aml_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
loss, acc = model.evaluate(test_data)
print("Frame-level Accuracy:", acc)


1243/1243 ━━━━━━━━━━━━━━━━━━━━ 731s 588ms/step - accuracy: 0.1577 - loss: 1.8037
Frame-level Accuracy: 0.1576550453901291


In [11]:
import numpy as np
from collections import defaultdict

video_preds = defaultdict(list)

for i in range(len(test_data.filenames)):
    img, label = test_data[i]
    preds = model.predict(img)

    for j, pred in enumerate(preds):
        filename = test_data.filenames[i * test_data.batch_size + j]
        video_name = filename.split("_")[0]   # extract video name
        video_preds[video_name].append(pred)

# Aggregate
final_preds = {}

for vid, preds in video_preds.items():
    avg_pred = np.mean(preds, axis=0)
    final_preds[vid] = np.argmax(avg_pred)

print("Video-level predictions:", final_preds)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13

ValueError: Asked to retrieve element 1243, but the Sequence has length 1243

In [12]:
from sklearn.metrics import classification_report

y_true = test_data.classes
y_pred = model.predict(test_data)
y_pred = y_pred.argmax(axis=1)

print(classification_report(y_true, y_pred))

1243/1243 ━━━━━━━━━━━━━━━━━━━━ 82s 66ms/step
              precision    recall  f1-score   support

           0       0.15      0.00      0.01      4781
           1       0.00      0.00      0.00      8679
           2       0.24      0.00      0.01      9075
           3       0.28      0.33      0.30     10793
           4       0.17      0.00      0.00      6417
           5       0.00      0.00      0.00         0

    accuracy                           0.09     39745
   macro avg       0.14      0.06      0.05     39745
weighted avg       0.17      0.09      0.09     39745



c:\Users\SwapnilShinde\OneDrive\Documents\Study Material\Sem 6\AML\Experiment Codes\aml_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SwapnilShinde\OneDrive\Documents\Study Material\Sem 6\AML\Experiment Codes\aml_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\SwapnilShinde\OneDrive\Documents\Study Material\Sem 6\AML\Experiment Codes\aml_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in la